# ORBIT — Phase 3: Continuous Real-Game Self-Play & Online Replay Augmentation

**Phase 3A:** Record 30 real Crafter episodes (<= 300 steps/ep) using Gate 6 H=4 policy candidate with exploration tau=1.2.
**Phase 3B:** Merge canonical baseline and self-play replay shards, extract frozen DINOv2 tokens, and audit sequence contiguity with prefix burn-in (T_prefix=4).
**Phase 3C:** Fine-tune world model on augmented replay buffer (Formulation C) and evaluate on 20 held-out seeds against Gate 4 baseline.

- Pure visual interface: . Zero privileged simulator variables.
- Strict seed disjointness against held-out evaluation seeds (3030..3049).
- Automated Google Drive backup of all replay shards, token caches, and model checkpoints.

---

### Cell 1: Environment & Hardware Diagnostics

In [ ]:
# Cell 1: Hardware Diagnostics
import os
import sys
import platform
import torch

print("=" * 75)
print("  SYSTEM & HARDWARE DIAGNOSTICS")
print("=" * 75)
print(f"Python Version:   {platform.python_version()} ({sys.executable})")
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Device Name: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM:       {vram_gb:.2f} GB")
else:
    print("Running on CPU mode. (Tip: For fast rollouts, select Runtime -> Change runtime type -> T4 GPU)")
print("=" * 75)

### Cell 2: Google Drive Mounting & Namespace Setup
Auto-detects Google Drive (`MyDrive` vs `My Drive`) and initializes the dedicated `learn-a-world/orbit/` persistent storage tree.

In [ ]:
# Cell 2: Robust Drive Mount & Namespace Isolation
import os
from pathlib import Path

# 1. Mount Google Drive safely
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive') and not os.path.exists('/content/drive/My Drive'):
        print("[Drive] Mounting Google Drive to /content/drive...")
        drive.mount('/content/drive')
    else:
        print("[Drive] Google Drive is already mounted.")
except ImportError:
    print("[Environment] Local or non-Colab environment detected.")

# 2. Detect Drive Root Path dynamically
if os.path.exists('/content/drive/MyDrive'):
    DRIVE_ROOT = Path('/content/drive/MyDrive/learn-a-world')
elif os.path.exists('/content/drive/My Drive'):
    DRIVE_ROOT = Path('/content/drive/My Drive/learn-a-world')
else:
    DRIVE_ROOT = Path('/content/learn-a-world/drive_backup')

# 3. Dedicated ORBIT Subdirectories
ORBIT_DRIVE = DRIVE_ROOT / 'orbit'
DRIVE_CHECKPOINTS = ORBIT_DRIVE / 'checkpoints'
DRIVE_DATA = ORBIT_DRIVE / 'data'
DRIVE_RESULTS = ORBIT_DRIVE / 'results'
DRIVE_LOGS = ORBIT_DRIVE / 'logs'

for p in [DRIVE_CHECKPOINTS, DRIVE_DATA, DRIVE_RESULTS, DRIVE_LOGS]:
    p.mkdir(parents=True, exist_ok=True)

print("[+] Google Drive configured successfully:")
print(f"    Master Drive Root:  {DRIVE_ROOT}")
print(f"    Orbit Checkpoints:  {DRIVE_CHECKPOINTS}")
print(f"    Orbit Data Shards:  {DRIVE_DATA}")

### Cell 3: Clean Repository Sync & Dependency Setup
Fetches the latest upstream commits cleanly (avoiding git merge/lock conflicts) and installs Crafter dependencies.

In [ ]:
# Cell 3: Repository Sync
%cd /content
import os
import sys

REPO_DIR = Path('/content/learn-a-world')
if (REPO_DIR / '.git').exists():
    %cd /content/learn-a-world
    !git fetch origin main
    !git reset --hard origin/main
else:
    !rm -rf /content/learn-a-world
    !git clone https://github.com/sahiltolani30/learn-a-world.git /content/learn-a-world
    %cd /content/learn-a-world

# Set python search path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.environ['PYTHONPATH'] = str(REPO_DIR)

# Install dependencies quietly
!pip install -q --upgrade pip
!pip install -q crafter gymnasium pytest numpy torchvision timm matplotlib
!pip install -q -e .

# Pre-clone DINOv2 to torch hub cache to prevent GitHub 504 Gateway Timeouts
!mkdir -p /root/.cache/torch/hub
!git clone --depth 1 https://github.com/facebookresearch/dinov2.git /root/.cache/torch/hub/facebookresearch_dinov2_main 2>/dev/null || (cd /root/.cache/torch/hub/facebookresearch_dinov2_main && git pull origin main)

print("[+] Repository synced to latest commit:")
!git log -1 --oneline

### Cell 4: Checkpoint Discovery & Verification
Verifies that the Gate 6 H=4 policy checkpoint is available (searches repository and Google Drive). If missing, provides an interactive one-click upload.

In [ ]:
# Cell 4: Checkpoint Verification
from pathlib import Path
import shutil

candidate_checkpoints = [
    Path('results/gate6/policy_h4.pt'),
    Path('results/gate6/policy_h4.pt.gz'),
    DRIVE_CHECKPOINTS / 'policy_h4.pt',
    DRIVE_CHECKPOINTS / 'policy_h4.pt.gz',
    DRIVE_ROOT / 'results/gate6/policy_h4.pt',
    DRIVE_ROOT / 'results/gate6/policy_h4.pt.gz',
]

active_checkpoint = None
for c in candidate_checkpoints:
    if c.exists() and c.is_file():
        active_checkpoint = c
        break

if active_checkpoint:
    print(f"[+] Gate 6 H=4 policy candidate verified at: {active_checkpoint}")
    print(f"    File Size: {active_checkpoint.stat().st_size / (1024*1024):.2f} MB")
    # Ensure a local copy exists in results/gate6/ for standard relative path loading
    Path('results/gate6').mkdir(parents=True, exist_ok=True)
    if not Path('results/gate6/policy_h4.pt').exists() and not Path('results/gate6/policy_h4.pt.gz').exists():
        shutil.copy2(str(active_checkpoint), f"results/gate6/{active_checkpoint.name}")
else:
    print("[-] policy_h4 checkpoint not found in repo or Google Drive.")
    print("    Please upload policy_h4.pt or policy_h4.pt.gz using the widget below:")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        dest = DRIVE_CHECKPOINTS / fname
        shutil.move(fname, str(dest))
        active_checkpoint = dest
        Path('results/gate6').mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(dest), f"results/gate6/{fname}")
        print(f"[+] Successfully saved uploaded checkpoint to: {dest}")

### Cell 5: Execute Phase 3A — Collect 30 Real Episodes & Validate Disk Reload
Runs the pure visual rollout collection, asserts transition schemas, validates boundary masks, and automatically writes the replay shard to both local disk and Google Drive.

In [ ]:
# Cell 5: Run Phase 3A
%cd /content/learn-a-world

!PYTHONPATH=/content/learn-a-world python scripts/run_phase3a_collection.py \
  --policy-checkpoint results/gate6/policy_h4.pt \
  --episodes 30 \
  --start-seed 1000 \
  --max-steps 300 \
  --temperature 1.2 \
  --device auto \
  --output data/phase3a_selfplay_30eps.npz \
  --drive-dir "{DRIVE_ROOT}"

### Cell 6: Visual Acceptance Report & Metrics Summary
Loads the verified report JSON and displays the acceptance checklist.

In [ ]:
# Cell 6: Display Phase 3A Acceptance Checklist
import json

report_path = Path('data/phase3a_verification_report.json')
if not report_path.exists():
    # Check Google Drive fallback
    report_path = DRIVE_DATA / 'phase3a_verification_report.json'

with open(report_path) as f:
    rep = json.load(f)

print("=" * 80)
print(f"PHASE 3A ACCEPTANCE REPORT: {rep['status']}")
print("=" * 80)
print(f"Episodes Collected:        {rep['num_episodes']}")
print(f"Total Frames:              {rep['total_frames']}")
print(f"Mean Return / Episode:     {rep['mean_return']:+.2f} ± {rep['std_return']:.2f}")
print(f"Mean Steps / Episode:      {rep['mean_steps']:.1f}")
print(f"Movement Percentage:       {rep['movement_percentage']:.1f}%")
print(f"Action Shannon Entropy:    {rep['action_entropy']:.3f}")
print(f"Replay File Size:          {rep['file_size_mb']:.2f} MB")
print(f"Local Replay Path:         {rep['output_file']}")
print(f"Drive Backup Path:         {DRIVE_DATA / 'phase3a_selfplay_30eps.npz'}")
print("-" * 80)
print("Acceptance Criteria Checklist:")
for item, val in rep['checklist'].items():
    print(f"  [{'PASS' if val else 'FAIL'}] {item.replace('_', ' ').capitalize()}")
print("=" * 80)

### Cell 7: Execute Phase 3B — Replay Buffer Integration & Sequence Verification
Merges the 54 baseline episodes (10,054 transitions) with the 30 self-play episodes (5,099 transitions) into an 84-episode augmented buffer. Extracts frozen DINOv2 ViT-S/14 patch tokens ($[T+1, 256, 384]$), checks sequence contiguity and prefix burn-in ($T_{\text{prefix}}=4$), confirms zero boundary bleed, and enforces strict held-out evaluation seed disjointness.

In [ ]:
# Cell 7: Run Phase 3B Integration
%cd /content/learn-a-world

!PYTHONPATH=/content/learn-a-world python scripts/run_phase3b_integration.py \
  --baseline-replay data/gate3/train_replay_10k.npz \
  --selfplay-replay data/phase3a_selfplay_30eps.npz \
  --heldout-eval data/gate3/eval_heldout_seeds.pt \
  --device auto \
  --output-dir data/phase3 \
  --drive-dir "{DRIVE_ROOT}"

### Cell 8: Visual Acceptance Report & Invariant Checklist for Phase 3B
Loads the verified Phase 3B report JSON and displays the acceptance checklist.

In [ ]:
# Cell 8: Display Phase 3B Acceptance Checklist
import json
from pathlib import Path

report_path = Path('data/phase3/phase3b_verification_report.json')
if not report_path.exists():
    report_path = DRIVE_DATA / 'phase3b_verification_report.json'

with open(report_path) as f:
    rep = json.load(f)

print("=" * 80)
print(f"PHASE 3B ACCEPTANCE REPORT: {rep['status']}")
print("=" * 80)
print(f"Baseline Episodes:           {rep['baseline_episodes']} ({rep['baseline_transitions']} transitions)")
print(f"Self-Play Episodes:          {rep['selfplay_episodes']} ({rep['selfplay_transitions']} transitions)")
print(f"Total Augmented Episodes:    {rep['total_augmented_episodes']} ({rep['total_augmented_transitions']} transitions)")
print(f"Total Observation Frames:    {rep['total_observation_frames']}")
print(f"Token Dimensions:            {rep['token_dimensions']}")
print(f"Augmented Replay Path:       {rep['augmented_replay_path']}")
print(f"Selfplay Token Cache:        {rep['selfplay_cache_path']}")
print(f"Combined Token Cache:        {rep['augmented_cache_path']}")
print("-" * 80)
print("Acceptance Criteria Checklist:")
for item, val in rep['checklist'].items():
    print(f"  [{'PASS' if val else 'FAIL'}] {item.replace('_', ' ').capitalize()}")
print("=" * 80)

### Cell 9: Execute Phase 3C — World Model Fine-Tuning & Diagnostic Evaluation
Fine-tunes the world model predictor, recurrent GRU belief core, and transition heads on the combined 84-episode augmented replay (Formulation C: change-weighted loss, autoregressive sequence length 8). Evaluates both the Gate 4 baseline and Phase 3 candidate on 20 held-out evaluation seeds (3030..3049) and verifies positive action sensitivity.

In [ ]:
# Cell 9: Run Phase 3C World Model Fine-Tuning & Evaluation
%cd /content/learn-a-world

!PYTHONPATH=/content/learn-a-world python scripts/run_phase3c_world_model.py \n  --base-checkpoint results/gate4/formulation_c_residual_change-weighted.pt.gz \n  --augmented-replay data/phase3/augmented_replay_84eps.npz \n  --augmented-token-cache data/phase3/augmented_token_cache.pt \n  --heldout-eval data/gate3/eval_heldout_seeds.pt.gz \n  --updates 500 \n  --batch-size 16 \n  --learning-rate 1e-4 \n  --device auto \n  --output-dir results/phase3 \n  --drive-dir "{DRIVE_ROOT}"

### Cell 10: Visual Acceptance Report & Diagnostic Plots for Phase 3C
Displays comparative evaluation metrics across 20 held-out seeds and renders diagnostic verification plots.

In [ ]:
# Cell 10: Display Phase 3C Acceptance Report & Plots
import json
from pathlib import Path
from IPython.display import Image, display

report_path = Path("results/phase3/phase3c_verification_report.json")
if not report_path.exists():
    report_path = DRIVE_RESULTS / "phase3c_verification_report.json"

with open(report_path) as f:
    rep = json.load(f)

print("=" * 80)
print(f"PHASE 3C ACCEPTANCE REPORT: {rep['status']} ({rep['promotion_decision']})")
print("=" * 80)
print(f"Selected Champion Model:     {rep['selected_model']}")
print(f"Fine-Tuning Updates:         {rep['updates_trained']} in {rep['training_time_sec']:.1f}s")
print(f"World Model Checkpoint:      {rep['checkpoint_saved']}")
print("-" * 80)
print("Diagnostic Comparison on 20 Held-Out Seeds:")
base = rep["baseline_gate4_metrics"]
cand = rep["candidate_phase3_metrics"]
print(f"  Overall Rel Error:         Gate 4: {base['one_step_relative_error']:.4f}  -->  Candidate: {cand['one_step_relative_error']:.4f}")
print(f"  Changed Token Rel Error:   Gate 4: {base['changed_token_relative_error']:.4f}  -->  Candidate: {cand['changed_token_relative_error']:.4f}")
print(f"  Static Token Rel Error:    Gate 4: {base['static_token_relative_error']:.4f}  -->  Candidate: {cand['static_token_relative_error']:.4f}")
print(f"  Action Sensitivity Gap:    Gate 4: {base['shuffled_action_sensitivity_gap']:+.4f}  -->  Candidate: {cand['shuffled_action_sensitivity_gap']:+.4f}")
print(f"  Free-Running H=4 MSE:      Gate 4: {base['free_running_errors']['4']:.4f}  -->  Candidate: {cand['free_running_errors']['4']:.4f}")
print("-" * 80)
print("Acceptance Criteria Checklist:")
for item, val in rep["checklist"].items():
    print(f"  [{'PASS' if val else 'FAIL'}] {item.replace('_', ' ').capitalize()}")
print("=" * 80)

plot_path = Path("results/phase3/phase3c_world_model_plots.png")
if not plot_path.exists():
    plot_path = DRIVE_RESULTS / "phase3c_world_model_plots.png"
if plot_path.exists():
    display(Image(str(plot_path)))
